In [1]:
import os
# 缓解显存碎片和过度预留的问题
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from tqdm import tqdm
import openai
import json
import time
import os
from PIL import Image

/root/miniconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 加载模型

In [3]:
from transformers import AutoModel, AutoTokenizer

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
ocr_model_path = "../model/deepseek-ocr"

tokenizer = AutoTokenizer.from_pretrained(ocr_model_path, _attn_implementation='flash_attention_2', trust_remote_code=True)
model = AutoModel.from_pretrained(
    ocr_model_path, trust_remote_code=True, use_safetensors=True
)
model = model.eval().cuda("cuda:0").to(torch.bfloat16)

# image_file = 'your_image.jpg'
# output_path = 'your/output/dir'

# infer(self, tokenizer, prompt='', image_file='', output_path = ' ', base_size = 1024, image_size = 640, crop_mode = True, test_compress = False, save_results = False):

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 640, image_size = 640, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# Gundam: base_size = 1024, image_size = 640, crop_mode = True

# res = model.infer(tokenizer, prompt=prompt, image_file=image_file, output_path = output_path, base_size = 1024, image_size = 640, crop_mode=True, save_results = True, test_compress = True)


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at ../model/deepseek-ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# 原论文中只评估了 tiny 和 small 模型 分别对应的image_size base_size = 512 640
def process_single_image(tokenizer, model, image_name, output_path, imgs_dir, mode,prompt):
    """
    处理单张图片，返回清理后的 OCR 文本
    """
    if mode == "tiny":
        IMAGE_SIZE = 512
        BASE_SIZE = 512
    elif mode == "small":
        IMAGE_SIZE = 640
        BASE_SIZE = 640
    elif mode == "raw":
        img = Image.open(os.path.join(imgs_dir, image_name))
        w, h = img.size
        long_side = max(w, h)
        IMAGE_SIZE = long_side
        BASE_SIZE = long_side
        
    image_path = os.path.join(imgs_dir, image_name)
    
    if mode == "raw":
        # 不使用压缩的 OCR 结果, 作为对比实验
        test_compress = False
    else:
        test_compress = True
        
    res = model.infer(
        tokenizer=tokenizer,
        prompt=prompt,
        image_file=image_path,
        output_path=output_path,
        base_size=BASE_SIZE,
        image_size=IMAGE_SIZE,
        # crop_mode=True,
        crop_mode=False,
        # save_results=True,    # 这个设置会将结果保存到output_path目录下
        save_results=False,
        eval_mode=True,         # 评估模式，不保存结果，将结果返回
        test_compress=test_compress,     # 使用压缩的 OCR 结果
    )
    
    return res

def vqa(tokenizer, model, data_path=None, output_path = "../output", save_path=None, imgs_dir=None, mode="tiny"):
    vqa_results = []
    image_names = [f"en_{i+1}.png" for i in range(112)]
    # TODO 仅测试用
    # image_names = image_names[:2]
    data = load_data(data_path)
    data_dict = {}
    for item in data:
        data_dict[item["image"]] = item
    # image_paths = [os.path.join(images_dir, img_name) for img_name in image_names]
    print(f"开始处理 {len(image_names)} 张图片...")
    # 进行vqa测试
    for image_name in tqdm(image_names):
        qa_pairs = data_dict[image_name]["qa_pairs"]
        for idx, qa in enumerate(qa_pairs):
            question = qa["question"]
            options = qa["options"]
            prompt = "<image>\nAnswer the question based on the image content. Only respond with the option letter (A/B/C/D).\nQuestion: " + question + "\nOptions: " + ", ".join(options) + "\nAnswer:" 
            LLMAnswer = process_single_image(tokenizer, model, image_name, output_path, imgs_dir, mode, prompt)
            qa["LLMAnswer"] = LLMAnswer
            print(f"处理图片 {image_name} 问题 {idx} 完成，LLM回答: {LLMAnswer} 正确答案: {qa['correct_answer']}")
        vqa_results.append(data_dict[image_name])

    # 按照image name重新排序
    vqa_results = sorted(
        vqa_results,
        key=lambda x: int(x["image"].split("_")[-1].split(".")[0])
    )
    
    # 将最终结果保存到文件中
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(vqa_results, f, ensure_ascii=False, indent=4)
    
    
    print(f"\n结果已保存到: {save_path}")
    

In [5]:
def batch_vqa(tokenizer, model, data_path=None, output_path = "../output", save_path=None, imgs_dir=None, mode="tiny"):
    vqa_results = []
    image_names = [f"en_{i+1}.png" for i in range(112)]
    # TODO 仅测试用
    image_names = image_names[:2]
    data = load_data(data_path)
    data_dict = {}
    for item in data:
        data_dict[item["image"]] = item
    # image_paths = [os.path.join(images_dir, img_name) for img_name in image_names]
    print(f"开始处理 {len(image_names)} 张图片...")
    # 进行vqa测试-批量推理
    test_dict = []
    prompts = []
    image_files = []
    all_qa_pairs = []
    for image_name in tqdm(image_names):
        qa_pairs = data_dict[image_name]["qa_pairs"]
        for idx, qa in enumerate(qa_pairs):
            question = qa["question"]
            options = qa["options"]
            prompt = "<image>\nAnswer the question based on the image content. Only respond with the option letter (A/B/C/D).\nQuestion: " + question + "\nOptions: " + ", ".join(options) + "\nAnswer:" 
            all_qa_pairs.append(qa)
            prompts.append(prompt)
            image_files.append( os.path.join(imgs_dir, image_name) )
        #     LLMAnswer = process_single_image(tokenizer, model, image_name, output_path, imgs_dir, mode, prompt)
        #     qa["LLMAnswer"] = LLMAnswer
        #     print(f"处理图片 {image_name} 问题 {idx} 完成，LLM回答: {LLMAnswer} 正确答案: {qa['correct_answer']}")
        # vqa_results.append(data_dict[image_name])

    # 批量推理
    res = model.infer_batch(
    tokenizer,
    prompts=prompts,
    image_files=image_files,
    base_size=512,
    image_size=512,
    crop_mode=False,
    save_results=False,
    eval_mode=True,
    )
    
    for i, qa in enumerate(all_qa_pairs):
        qa["LLMAnswer"] = res[i]
    
    # 将结果更新到原数据中
    index = 0
    for image_name in image_names:
        qa_pairs = data_dict[image_name]["qa_pairs"]
        for qa in qa_pairs:
            qa["LLMAnswer"] = res[index]
            index += 1
    
    vqa_results = [data_dict[image_name] for image_name in image_names]
    
    # 将最终结果保存到文件中
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(vqa_results, f, ensure_ascii=False, indent=4)
    
    
    print(f"\n结果已保存到: {save_path}")

## VQA

In [6]:
data_path = "../fox_data/qa/qa_recheck.json"

In [7]:
# vqa(tokenizer, model, data_path, "../output", save_path="../results/vqa/en_png_tiny.json", imgs_dir="../fox_data/en_png", mode="tiny")

In [ ]:
# vqa(tokenizer, model, data_path, "../output", save_path="../results/vqa/en_png_tiny.json", imgs_dir="../fox_data/en_png", mode="tiny")

开始处理 112 张图片...


  0%|          | 0/112 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES
处理图片 en_1.png 问题 0 完成，LLM回答: B
Explanation: The head of a public body has 15 business days to respond to a written appeal under subsection (1)(a), excluding any extension. 正确答案: A
directly resize


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 64, 1280])
NO PATCHES


...

处理图片 en_112.png 问题 0 完成，LLM回答: D
Explanation: The correct answer is D, as the Administration of Estates Act, 1965 (Act No. 66 of 1965) governs the administration of estates in the UK. 正确答案: B
directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


处理图片 en_112.png 问题 1 完成，LLM回答: C. The Native Administration Proclamation, 1928 did not provide for the transfer of estates administered under the Native Administration Proclamation, 1928 to the administration of the Native Affairs Department. 正确答案: C
directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


100%|██████████| 112/112 [2:42:11<00:00, 86.89s/it]

处理图片 en_112.png 问题 2 完成，LLM回答: C. The Master's functions can be assigned to magistrates under section 4A(1). 正确答案: D

结果已保存到: ../results/vqa/en_png_tiny.json


In [ ]:
# vqa(tokenizer, model, data_path, "../output", save_path="../results/vqa/distort_tiny.json", imgs_dir="../fox_data/distort", mode="tiny")

开始处理 112 张图片...


  0%|          | 0/112 [00:00<?, ?it/s]

/root/miniconda3/envs/py311/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


...

处理图片 en_7.png 问题 1 完成，LLM回答: D 正确答案: C
directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


  6%|▋         | 7/112 [05:24<58:32, 33.45s/it]  The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


处理图片 en_7.png 问题 2 完成，LLM回答: C. To track progress towards the completion of a project. 正确答案: B
directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


处理图片 en_8.png 问题 0 完成，LLM回答: C
Explanation: The life expectancy table must be used to calculate the required minimum distribution under paragraphs 3(a) and 3(b)(i) for years following the depositor’s death. The table is used to calculate the required minimum distribution for the life expectancy of the deceased person, which is calculated based on the life expectancy of the deceased person and the life expectancy of the depositor. The table is used to calculate the required minimum distribution for the life expectancy of the deceased person, which is calculated based on the life expectancy of the deceased person and the age of the deceased person. The table is used to calculate the required minimum distribution for the life expectancy of the deceased person, which is calculated based on the life expectancy of the deceased person and the date of the deceased person’s death. The table is used to calculate the required minimum distribution for the life expectancy of the deceased person, wh

In [7]:
vqa(tokenizer, model, data_path, "../output", save_path="../results/vqa/from_text_tiny.json", imgs_dir="../fox_data/from_text", mode="tiny")

开始处理 112 张图片...


  0%|          | 0/112 [00:00<?, ?it/s]

directly resize


/root/miniconda3/envs/py311/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
The attention layers in this model are transitioning from computing the RoPE e

BASE:  torch.Size([1, 64, 1280])
NO PATCHES


...

处理图片 en_112.png 问题 0 完成，LLM回答: C. The Administration of Estates (Rehoboth Gebiet) Proclamation, 1941, must be liquidated and distributed and any matter relating to the liquidation and distribution of such estate must be dealt with as if this Act had not been passed. (3) Despite subsection (2), if, in the case of an estate referred to in that subsection the liquidation and distribution of which immediately before the commencement of this Act was being administered in terms of the Native Administration Proclamation, 1928 or the Administration of Estates (Rehoboth Gebiet) Proclamation, 1941 and which has not been completed by the date of such commencement, any person having an interest in the estate in writing requests the Master of the High Court to administer the estate in question in terms of the Administration of Estates Act, 1965, the Master must (a) in writing request the magistrate in charge of the supervision of the estate in question to transfer such estate to the Master's superv

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 64, 1280])
NO PATCHES


100%|██████████| 112/112 [1:48:13<00:00, 57.98s/it]

处理图片 en_112.png 问题 2 完成，LLM回答: C. The High Court 正确答案: D

结果已保存到: ../results/vqa/from_text_tiny.json


In [8]:
vqa(tokenizer, model, data_path, "../output", save_path="../results/vqa/from_text_raw.json", imgs_dir="../fox_data/from_text", mode="raw")

开始处理 112 张图片...


  0%|          | 0/112 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 196, 1280])
NO PATCHES
处理图片 en_1.png 问题 0 完成，LLM回答: D 正确答案: A


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 196, 1280])
NO PATCHES
处理图片 en_1.png 问题 1 完成，LLM回答: C. Issue a writ of certiorari to quash the disclosure order 正确答案: B


...

BASE:  torch.Size([1, 196, 1280])
NO PATCHES


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


处理图片 en_112.png 问题 1 完成，LLM回答: D
Explanation: The text states that estates administered under the Native Administration Proclamation, 1928, were not affected by the 2005 Act. 正确答案: C
BASE:  torch.Size([1, 196, 1280])
NO PATCHES


100%|██████████| 112/112 [59:47<00:00, 32.03s/it]

处理图片 en_112.png 问题 2 完成，LLM回答: C. The Master of the Court 正确答案: D

结果已保存到: ../results/vqa/from_text_raw.json
